<a href="https://colab.research.google.com/github/Innovatewithapple/NeuralNetwork/blob/main/TransformerPractice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input,Dense,Embedding,Layer
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
subjects = ["I", "You", "He", "She", "We", "They"]
verbs = ["eat", "like", "see", "have", "want", "buy", "find", "need"]
objects = ["apple", "car", "book", "coffee", "house", "dog", "cat", "food"]

# English → Spanish mapping (simple)
translation = {
    "I": "Yo", "You": "Tú", "He": "Él", "She": "Ella", "We": "Nosotros", "They": "Ellos",
    "eat": "como", "like": "gusta", "see": "veo", "have": "tengo",
    "want": "quiero", "buy": "compro", "find": "encuentro", "need": "necesito",
    "apple": "manzana", "car": "coche", "book": "libro", "coffee": "café",
    "house": "casa", "dog": "perro", "cat": "gato", "food": "comida"
}

input_texts = []
target_texts = []

for s in subjects:
    for v in verbs:
        for o in objects:
            eng = f"{s} {v} {o}"
            spa = f"{translation[s]} {translation[v]} {translation[o]}"
            input_texts.append(eng)
            target_texts.append(spa)

print(len(input_texts))  # 6 * 8 * 8 = 384

384


In [3]:
extra_phrases = [
    ("Good morning", "Buenos días"),
    ("Thank you", "Gracias"),
    ("I am happy", "Estoy feliz"),
    ("Where is the car", "Dónde está el coche"),
    ("I need help", "Necesito ayuda"),
    ("See you soon", "Hasta pronto"),
    ("I am learning", "Estoy aprendiendo"),
    ("The sky is blue", "El cielo es azul"),
    ("The water is cold", "El agua está fría"),
    ("It is hot today", "Hace calor hoy")
]

for eng, spa in extra_phrases:
    input_texts.append(eng)
    target_texts.append(spa)

In [4]:
input_texts = input_texts
target_texts = target_texts

target_texts = ["startseq " + t + " endseq" for t in target_texts]

In [5]:
#Tokenizer
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

output_tokenizer = Tokenizer()
output_tokenizer.fit_on_texts(target_texts)
output_sequences = output_tokenizer.texts_to_sequences(target_texts)

In [6]:
input_vocab_size = len(input_tokenizer.word_index) + 1
output_vocab_size = len(output_tokenizer.word_index) + 1

In [7]:
max_input_length = max(len(seq) for seq in input_sequences)
max_output_length = max(len(seq) for seq in output_sequences)

input_sequences = pad_sequences(input_sequences,maxlen=max_input_length,padding='post')
output_sequences = pad_sequences(output_sequences,maxlen=max_output_length,padding='post')

In [8]:
decoder_input = output_sequences[:,:-1]
decoder_output = output_sequences[:,1:]

decoder_output = np.expand_dims(decoder_output,-1)

In [9]:
class Attention(Layer):
  def __init__(self,d_model):
      super().__init__()
      self.Wq = Dense(d_model)
      self.Wk = Dense(d_model)
      self.Wv = Dense(d_model)

  def call(self,q,k,v):
    Q = self.Wq(q)
    K = self.Wk(k)
    V = self.Wv(v)

    scores = tf.matmul(Q,K,transpose_b=True)
    dk = tf.cast(tf.shape(K)[-1],tf.float32)
    scores = scores / tf.sqrt(dk)

    weights = tf.nn.softmax(scores)
    output = tf.matmul(weights,V)

    return output


In [11]:
encoder_input = Input(shape=(max_input_length,))
x = Embedding(input_vocab_size,64)(encoder_input)

encoder_output = Attention(64)(x,x,x)

decoder_inputs = Input(shape=(max_output_length - 1,))
y = Embedding(output_vocab_size,64)(decoder_inputs)

decoder_outputs = Attention(64)(y,y,y)

# self attention with cross attention
final_decoder = Attention(64)(decoder_outputs,encoder_output,encoder_output)

output = Dense(output_vocab_size,activation='softmax')(final_decoder)

model = Model([encoder_input,decoder_inputs],output)

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit([input_sequences,decoder_input],decoder_output,epochs=200,validation_split=0.2)


Epoch 1/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 58ms/step - accuracy: 0.1511 - loss: 3.7579 - val_accuracy: 0.1924 - val_loss: 3.7155
Epoch 2/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2000 - loss: 3.5335 - val_accuracy: 0.1949 - val_loss: 3.4979
Epoch 3/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2000 - loss: 3.0976 - val_accuracy: 0.2000 - val_loss: 3.3090
Epoch 4/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2000 - loss: 2.9022 - val_accuracy: 0.2101 - val_loss: 3.3156
Epoch 5/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2000 - loss: 2.7951 - val_accuracy: 0.1924 - val_loss: 3.3560
Epoch 6/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2000 - loss: 2.7459 - val_accuracy: 0.2025 - val_loss: 3.4279
Epoch 7/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2000 - loss: 2.7083 - val_accuracy: 0.2000 - val_loss: 3.4998
Epoch 8/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2000 - loss: 2.6750 - val_accuracy: 0.